# Step 1 – U-Net flood baseline on Sen1Floods11
Trains a U-Net on the 446 hand-labelled Sentinel-1 chips (VV, VH) and reports flood/water IoU on:
- the standard **test** split
- the **Bolivia** split (an event never seen in training = a first transferability test)

Run in Google Colab: *Runtime → Change runtime type → GPU (T4)*. Full run ≈ 20–30 min.

In [ ]:
!pip -q install segmentation-models-pytorch rasterio
import os, glob, random, numpy as np, pandas as pd, rasterio, torch, torch.nn as nn
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
torch.manual_seed(0); random.seed(0); np.random.seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV)

## 1. Download the hand-labelled data (~1 GB)

In [ ]:
ROOT = '/content/sen1floods11'
os.makedirs(ROOT, exist_ok=True)
B = 'gs://sen1floods11/v1.1'
!gsutil -m -q cp -r {B}/data/flood_events/HandLabeled/S1Hand {ROOT}/
!gsutil -m -q cp -r {B}/data/flood_events/HandLabeled/LabelHand {ROOT}/
!gsutil -m -q cp -r {B}/splits/flood_handlabeled {ROOT}/
print(len(glob.glob(f'{ROOT}/S1Hand/*.tif')), 'S1 chips')
print(os.listdir(f'{ROOT}/flood_handlabeled'))

## 2. Read the official splits

In [ ]:
def chip_ids(csv):
    df = pd.read_csv(csv, header=None)
    return [f.replace('_S1Hand.tif', '') for f in df[0]]

S = f'{ROOT}/flood_handlabeled'
split = {k: chip_ids(f'{S}/flood_{k}_data.csv') for k in ['train', 'valid', 'test', 'bolivia']}
for k, v in split.items(): print(k, len(v))

## 3. Dataset
Labels: -1 = no data (ignored), 0 = dry, 1 = water. SAR is clipped to [-50, 1] dB and standardised with training-set stats.

In [ ]:
def load(cid):
    with rasterio.open(f'{ROOT}/S1Hand/{cid}_S1Hand.tif') as s: x = s.read().astype('float32')
    with rasterio.open(f'{ROOT}/LabelHand/{cid}_LabelHand.tif') as l: y = l.read(1).astype('int64')
    x = np.clip(np.nan_to_num(x, nan=-50), -50, 1)
    y[y < 0] = 255                      # ignore index
    return x, y

data = {cid: load(cid) for k in split for cid in split[k]}
tr = np.stack([data[c][0] for c in split['train']])
MEAN, STD = tr.mean((0, 2, 3)), tr.std((0, 2, 3))
print('VV/VH mean', MEAN, 'std', STD)

class S1(Dataset):
    def __init__(self, ids, train=False, crop=256):
        self.ids, self.train, self.crop = ids, train, crop
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        x, y = data[self.ids[i]]
        x = (x - MEAN[:, None, None]) / STD[:, None, None]
        if self.train:                      # random crop + flips
            r, c = np.random.randint(0, 512 - self.crop, 2)
            x, y = x[:, r:r+self.crop, c:c+self.crop], y[r:r+self.crop, c:c+self.crop]
            if random.random() < .5: x, y = x[:, :, ::-1], y[:, ::-1]
            if random.random() < .5: x, y = x[:, ::-1], y[::-1]
        return torch.tensor(x.copy()), torch.tensor(y.copy())

dl = {k: DataLoader(S1(v, train=(k == 'train')), batch_size=16 if k == 'train' else 4,
                    shuffle=(k == 'train'), num_workers=2) for k, v in split.items()}

## 4. Model and training

In [ ]:
model = smp.Unet('resnet34', encoder_weights=None, in_channels=2, classes=2).to(DEV)
# water is the minority class -> weight it up
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1., 3.], device=DEV), ignore_index=255)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 60
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)

@torch.no_grad()
def evaluate(loader):
    model.eval(); tp = fp = fn = tn = 0
    for x, y in loader:
        p = model(x.to(DEV)).argmax(1).cpu(); m = y != 255
        p, y = p[m], y[m]
        tp += ((p == 1) & (y == 1)).sum().item(); fp += ((p == 1) & (y == 0)).sum().item()
        fn += ((p == 0) & (y == 1)).sum().item(); tn += ((p == 0) & (y == 0)).sum().item()
    iou = tp / (tp + fp + fn + 1e-9); prec = tp / (tp + fp + 1e-9); rec = tp / (tp + fn + 1e-9)
    return dict(IoU=iou, F1=2*prec*rec/(prec+rec+1e-9), precision=prec, recall=rec,
                OA=(tp+tn)/(tp+tn+fp+fn))

best = 0
for ep in range(EPOCHS):
    model.train(); tot = 0
    for x, y in dl['train']:
        x, y = x.to(DEV), y.to(DEV)
        opt.zero_grad(); l = loss_fn(model(x), y); l.backward(); opt.step(); tot += l.item()
    sched.step()
    v = evaluate(dl['valid'])
    if v['IoU'] > best: best = v['IoU']; torch.save(model.state_dict(), 'unet_best.pt')
    if ep % 5 == 0 or ep == EPOCHS - 1:
        print(f"epoch {ep:2d}  loss {tot/len(dl['train']):.3f}  val IoU {v['IoU']:.3f}")

## 5. Results (use these numbers as your baseline)

In [ ]:
model.load_state_dict(torch.load('unet_best.pt'))
res = pd.DataFrame({k: evaluate(dl[k]) for k in ['valid', 'test', 'bolivia']}).T.round(3)
res.to_csv('baseline_unet_results.csv'); res

## 6. Look at predictions

In [ ]:
ds = S1(split['bolivia']); model.eval()
fig, ax = plt.subplots(3, 3, figsize=(10, 10))
for r, i in enumerate(random.sample(range(len(ds)), 3)):
    x, y = ds[i]
    with torch.no_grad(): p = model(x[None].to(DEV)).argmax(1)[0].cpu()
    ax[r, 0].imshow(x[0], cmap='gray'); ax[r, 1].imshow(np.ma.masked_equal(y, 255), vmin=0, vmax=1)
    ax[r, 2].imshow(p, vmin=0, vmax=1)
for a, t in zip(ax[0], ['VV (standardised)', 'Label', 'Prediction']): a.set_title(t)
for a in ax.flat: a.axis('off')
plt.tight_layout()

## Next
1. Save `baseline_unet_results.csv` – it is your reference row in the paper's results table.
2. Add HAND + slope as extra channels (needs DEM chips) → compare with this baseline.
3. Swap in `smp.Unet(..., decoder_attention_type='scse')` as the attention variant.